<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/HES16Sim008.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
D_test = [3, 5]
results = {}
alpha = 0.2
beta = 0.001
steps = 50
num_seeds = 5

# --- Localized seed initializer ---
def initialize_with_seeds(D, N, seed=0):
    rng = np.random.default_rng(seed)
    s = np.zeros([N]*D)
    center = tuple([N//2]*D)
    s[center] = 1.0  # single spike
    s += 0.01 * rng.standard_normal([N]*D)  # small noise
    return s

# --- Laplacian evolution ---
def evolve_field(s, alpha=0.2, beta=0.001, steps=50, clip=10.0):
    for _ in range(steps):
        lap = np.zeros_like(s)
        for a in range(s.ndim):
            lap += np.roll(s, 1, axis=a) + np.roll(s, -1, axis=a) - 2*s
        s = s + alpha*lap - beta*(s**3)
        s = np.clip(s, -clip, clip)
        s -= s.mean()
    return s

# --- λ* estimator with DC masking ---
def measure_lambda_with_DC_masking(field, dx=1.0):
    F = np.fft.fftn(field)
    P = np.abs(F)**2
    P = np.fft.fftshift(P)
    k_axes = [np.fft.fftshift(2*np.pi*np.fft.fftfreq(n, d=dx)) for n in field.shape]
    K = np.meshgrid(*k_axes, indexing='ij')
    center = tuple(n//2 for n in P.shape)
    P[center] = 0.0  # mask DC
    peak = np.unravel_index(np.argmax(P), P.shape)
    k_mag = np.sqrt(sum(Ki[peak]**2 for Ki in K))
    return (2*np.pi/k_mag) if k_mag > 0 else np.inf

# --- Run ensemble for D=3,5 ---
for D in D_test:
    N = 24 if D == 3 else 16
    lam_ensemble = []
    for seed in range(num_seeds):
        s = initialize_with_seeds(D, N, seed=seed)
        s = evolve_field(s, alpha=alpha, beta=beta, steps=steps)
        lam = measure_lambda_with_DC_masking(s)
        lam_ensemble.append(lam)
    lam_array = np.array(lam_ensemble)
    results[D] = {
        'mean': lam_array.mean(),
        'std': lam_array.std(),
        'ci95': 1.96 * lam_array.std() / np.sqrt(num_seeds)
    }

# --- Ratio test ---
lam3 = results[3]['mean']
lam5 = results[5]['mean']
ratio_measured = lam5 / lam3
ratio_predicted = np.sqrt(3/5)

print(f"λ*(D=3) ≈ {lam3:.3f} ± {results[3]['ci95']:.3f}")
print(f"λ*(D=5) ≈ {lam5:.3f} ± {results[5]['ci95']:.3f}")
print(f"Ratio λ₅ / λ₃ = {ratio_measured:.3f} vs predicted √(3/5) = {ratio_predicted:.3f}")


λ*(D=3) ≈ 1.187 ± 0.019
λ*(D=5) ≈ 0.946 ± 0.024
Ratio λ₅ / λ₃ = 0.797 vs predicted √(3/5) = 0.775
